In [1]:
!pip install transformers datasets torch accelerate

In [2]:
sample_text = """
Artificial intelligence is revolutionizing industries across the globe.
Machine learning enables systems to learn from data without being explicitly programmed.
Deep learning uses multi-layered neural networks to extract complex patterns.
Natural language processing allows machines to understand and generate human language.
GPT-2 is a powerful autoregressive language model developed by OpenAI.
Transformer models use self-attention mechanisms to capture long-range dependencies.
Fine-tuning adapts a pretrained model to a specific domain or task.
Transfer learning leverages knowledge from one task to improve performance on another.
Text generation models predict the next token based on previous context.
The attention mechanism allows the model to focus on relevant parts of the input.
Training large language models requires significant computational resources.
Tokenization is the process of converting text into numerical representations.
The vocabulary of GPT-2 consists of 50,257 unique tokens.
Beam search and sampling are common decoding strategies for text generation.
Temperature controls the randomness of the generated text output.
Top-k sampling restricts generation to the k most probable next tokens.
Nucleus sampling selects from the smallest set of tokens whose probabilities sum to p.
Perplexity is a common metric used to evaluate language model performance.
Lower perplexity indicates the model is more confident in its predictions.
Overfitting occurs when a model performs well on training but poorly on new data.
Regularization techniques such as dropout help prevent overfitting in deep networks.
Batch normalization stabilizes training by normalizing layer inputs.
The Adam optimizer adapts learning rates for each parameter during training.
Learning rate scheduling improves convergence by adjusting the rate during training.
Data augmentation increases the diversity of training data to improve generalization.
Neural networks are inspired by the structure and function of the human brain.
Convolutional neural networks excel at processing grid-structured data like images.
Recurrent neural networks are designed to handle sequential data over time.
Long short-term memory networks solve the vanishing gradient problem in RNNs.
The encoder-decoder architecture is widely used in sequence-to-sequence tasks.
BERT uses bidirectional context to produce rich text representations.
GPT models are unidirectional and trained using causal language modeling.
Pre-training on large corpora gives models broad language understanding.
Zero-shot learning allows models to perform tasks without any task-specific training.
Few-shot learning requires only a small number of examples to adapt to new tasks.
Prompt engineering designs effective inputs to guide language model outputs.
Reinforcement learning from human feedback improves model alignment and safety.
Model compression techniques like pruning reduce model size for deployment.
Knowledge distillation transfers knowledge from a large model to a smaller one.
Explainability and interpretability are critical for responsible AI deployment.
Bias in training data can lead to unfair or harmful model outputs.
Ethical AI frameworks guide the responsible development and use of AI systems.
Open source tools like Hugging Face Transformers accelerate AI research.
Cloud platforms like Google Colab provide free GPU access for model training.
Version control with Git helps manage code changes in collaborative projects.
Experiment tracking tools like Weights and Biases log training metrics efficiently.
Model evaluation should include both quantitative metrics and qualitative analysis.
Continuous integration ensures that code changes do not break existing functionality.
Documentation is essential for reproducibility and collaboration in AI projects.
""" * 30

with open("train.txt", "w") as f:
    f.write(sample_text)

word_count = len(sample_text.split())
print(f" Dataset created!")
print(f" Total words: {word_count}")
print(f" Total characters: {len(sample_text)}")

 Dataset created!
 Total words: 15750
 Total characters: 114810


In [3]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from transformers import DataCollatorForLanguageModeling
from transformers import Trainer, TrainingArguments
from torch.utils.data import Dataset
import torch
import math

class GPT2Dataset(Dataset):
    def __init__(self, file_path, tokenizer, block_size=64):
        with open(file_path, "r") as f:
            text = f.read()
        tokenized = tokenizer(
            text,
            return_tensors="pt",
            truncation=False,
            padding=False
        )
        input_ids = tokenized["input_ids"].squeeze()
        self.examples = []
        for i in range(0, len(input_ids) - block_size + 1, block_size):
            self.examples.append(input_ids[i : i + block_size])

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return {"input_ids": self.examples[idx], "labels": self.examples[idx]}


print("🔄 Loading model...")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

train_dataset = GPT2Dataset("train.txt", tokenizer, block_size=64)
print(f" Dataset: {len(train_dataset)} chunks ready for training")

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

training_args = TrainingArguments(
    output_dir="./gpt2-finetuned",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=50,
    save_steps=500,
    learning_rate=5e-5,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
)

print(" Starting training...")
train_result = trainer.train()

loss = train_result.training_loss
perplexity = math.exp(loss)
print(f"\n{'='*40}")
print(f" Training Complete!")
print(f" Final Loss     : {loss:.4f}")
print(f" Perplexity     : {perplexity:.2f}")
print(f"⏱  Total Steps    : {train_result.global_step}")
print(f"{'='*40}")

trainer.save_model("./gpt2-finetuned")
print(" Model saved!")

🔄 Loading model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Step,Training Loss
50,3.789616
100,1.575349
150,0.361269
200,0.182288
250,0.148110
300,0.117579
350,0.109997
400,0.096262


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


 Training Complete!
 Final Loss     : 0.7488
 Perplexity     : 2.11
⏱  Total Steps    : 430


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

💾 Model saved!


In [4]:
from transformers import pipeline, GPT2Tokenizer
import textwrap

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

generator = pipeline(
    "text-generation",
    model="./gpt2-finetuned",
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

prompts = [
    "Artificial intelligence is",
    "Deep learning models can",
    "The future of machine learning",
    "Natural language processing helps"
]

print("=" * 60)
print("         GPT-2 FINE-TUNED TEXT GENERATION")
print("=" * 60)

for prompt in prompts:
    result = generator(
        prompt,
        max_length=100,
        num_return_sequences=1,
        temperature=0.7,
        top_p=0.95,
        top_k=50,
        repetition_penalty=1.2,
        do_sample=True
    )
    print(f"\n Prompt: '{prompt}'")
    print("-" * 60)
    wrapped = textwrap.fill(result[0]["generated_text"], width=60)
    print(wrapped)
    print("-" * 60)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'num_return_sequences', 'top_k', 'top_p', 'temperature', 'do_sample', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


         GPT-2 FINE-TUNED TEXT GENERATION


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



 Prompt: 'Artificial intelligence is'
------------------------------------------------------------
Artificial intelligence is revolutionizing industries across
the globe. Machine learning enables systems to learn from
data without being explicitly programmed. Deep neural
networks are designed to extract complex patterns
efficiently. Natural language processing allows machines
understand and generate human languages. GPT-2 is a powerful
autoregressive text generation model developed by OpenAI,
based on feedback mechanisms such as pruning reduce model
size for deployment. Transformer models use self-attention
mechanism to capture long-range dependencies. Fine-tuning
adapts a pretrained part of its output into new tasks. BERT
uses bidirectional context to produce rich representations.
Text generation designs effective inputs to guide language
model outputs. Model compression techniques like pruned
trees improve convergence by adjusting the rate during
training. Knowledge distillation tra

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



 Prompt: 'Deep learning models can'
------------------------------------------------------------
Deep learning models can extract complex patterns. Natural
language processing allows machines to understand and
generate human vocabulary. GPT-2 is a powerful
autoregressive text generation model developed by OpenAI
with support from Google Colab.  The Adam optimizer adapts
learned rates for each parameter during training. During
normalization, the rate increases as input data grows
larger. Increasingly trained AI systems are designed to
handle sequential inputs over time using temporal
dependencies. Shortcoming mechanisms such like dropout help
prevent overfitting in deep networks.
------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



 Prompt: 'The future of machine learning'
------------------------------------------------------------
The future of machine learning requires only a small number
more examples to adapt to new tasks. Prompt engineering
designs effective inputs to guide language model outputs.
Reinforcement learning from human feedback improves models'
generalization efficiency. Model compression techniques like
pruning reduce model size for deployment.
------------------------------------------------------------

 Prompt: 'Natural language processing helps'
------------------------------------------------------------
Natural language processing helps prevent overfitting in
deep networks. Batch normalization stabilizes training by
Normalizing layer inputs. The Adam optimizer adapts learning
rates for each parameter during training. Learning rate
scheduling improves convergence when the cost of labor is
low. Data augmentation increases the diversity and
functionability to improve generalisation. Neural 

In [5]:
import gradio as gr
from transformers import pipeline, GPT2Tokenizer
import torch

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
generator = pipeline(
    "text-generation",
    model="./gpt2-finetuned",
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

def generate_text(prompt, max_length, temperature, top_p):
    result = generator(
        prompt,
        max_length=int(max_length),
        temperature=float(temperature),
        top_p=float(top_p),
        repetition_penalty=1.2,
        do_sample=True,
        num_return_sequences=1
    )
    return result[0]["generated_text"]

demo = gr.Interface(
    fn=generate_text,
    inputs=[
        gr.Textbox(label="Enter your prompt", placeholder="Artificial intelligence is..."),
        gr.Slider(50, 300, value=100, label="Max Length"),
        gr.Slider(0.1, 1.0, value=0.7, label="Temperature (creativity)"),
        gr.Slider(0.1, 1.0, value=0.95, label="Top-p (diversity)"),
    ],
    outputs=gr.Textbox(label="Generated Text", lines=8),
    title="🤖 GPT-2 Fine-Tuned Text Generator",
    description="Enter a prompt and adjust settings to generate AI text!",
    examples=[
        ["Artificial intelligence is", 100, 0.7, 0.95],
        ["Deep learning models can", 120, 0.8, 0.90],
        ["The future of machine learning", 150, 0.6, 0.95],
    ]
)

demo.launch(share=True)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6f2cb9d3d474802666.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
